# Deepfake Detection — Kaggle Full Training

This notebook handles:
1. Auto-detecting Celeb-DF-v2 under any nested Kaggle path
2. Symlinking videos into `data/raw/{real,fake}/`
3. Creating `data/processed/metadata.json`
4. Installing dependencies
5. Full training with checkpoint-based resume

> **Before running:** Add the Celeb-DF-v2 dataset via *Add Data* in the Kaggle UI.

## Cell 1 — Verify GPU

In [ ]:
import subprocess, os, sys
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found')

## Cell 2 — Detect Celeb-DF-v2 Root (Robust)

Handles any nesting level at `/kaggle/input`.

In [ ]:
from pathlib import Path

# ---- 1. List what datasets were attached ----
kaggle_input = Path('/kaggle/input')
print('Attached datasets:')
for p in sorted(kaggle_input.iterdir()):
    print(f'  {p}')

# ---- 2. Recursively locate Celeb-DF-v2 root ----
# The root is identified by containing BOTH 'Celeb-real' and 'Celeb-synthesis'
def find_celebdf_root(base: Path, max_depth: int = 5) -> Path | None:
    """Walk directories up to max_depth looking for Celeb-DF-v2 root."""
    if max_depth == 0:
        return None
    for child in base.iterdir():
        if not child.is_dir():
            continue
        # Direct match
        if (child / 'Celeb-real').exists() and (child / 'Celeb-synthesis').exists():
            return child
        # Also accept if current dir IS the root
        if child.name in ('Celeb-real', 'Celeb-synthesis'):
            parent = child.parent
            if (parent / 'Celeb-real').exists() and (parent / 'Celeb-synthesis').exists():
                return parent
        # Recurse
        result = find_celebdf_root(child, max_depth - 1)
        if result:
            return result
    return None

celebdf_root = find_celebdf_root(kaggle_input)

if celebdf_root is None:
    raise RuntimeError(
        'Could not find Celeb-DF-v2 root.\n'
        'Expected a folder containing both Celeb-real/ and Celeb-synthesis/.\n'
        f'Searched under: {kaggle_input}\n'
        'Please attach the dataset via Add Data > Datasets > celeb-df-v2'
    )

print(f'\n[OK] Found Celeb-DF-v2 at: {celebdf_root}')
for sub in ['Celeb-real', 'YouTube-real', 'Celeb-synthesis']:
    p = celebdf_root / sub
    count = len(list(p.glob('*.mp4'))) if p.exists() else 'NOT FOUND'
    print(f'  {sub}: {count} mp4 files')

## Cell 3 — Clone / Pull Repo

In [ ]:
import subprocess

REPO_URL = 'https://github.com/Harshvoidmain/deepfake-detection-multimodal-av.git'
REPO_DIR = '/kaggle/working/deepfake-detection-multimodal-av'

if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

print(f'[OK] Repo at {REPO_DIR}')
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

## Cell 4 — Install Dependencies

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
    check=True
)
print('[OK] Dependencies installed')

## Cell 5 — Symlink Dataset into data/raw/

In [ ]:
import os
from pathlib import Path

# Ensure working dir is the repo
os.chdir(REPO_DIR)

real_dst = Path('data/raw/real')
fake_dst = Path('data/raw/fake')
real_dst.mkdir(parents=True, exist_ok=True)
fake_dst.mkdir(parents=True, exist_ok=True)

def symlink_videos(src_dir: Path, dst_dir: Path, prefix: str = '') -> int:
    """Create symlinks for all .mp4 files from src into dst."""
    count = 0
    if not src_dir.exists():
        print(f'  [SKIP] {src_dir} not found')
        return 0
    for video in src_dir.glob('*.mp4'):
        name = f'{prefix}{video.name}' if prefix else video.name
        link = dst_dir / name
        if not link.exists():
            link.symlink_to(video.resolve())
        count += 1
    return count

# Real sources
real_count = symlink_videos(celebdf_root / 'Celeb-real',    real_dst, prefix='Celeb-real_')
real_count += symlink_videos(celebdf_root / 'YouTube-real', real_dst, prefix='YouTube-real_')

# Fake source
fake_count = symlink_videos(celebdf_root / 'Celeb-synthesis', fake_dst)

print(f'[OK] Symlinked {real_count} real videos → data/raw/real/')
print(f'[OK] Symlinked {fake_count} fake videos → data/raw/fake/')

## Cell 6 — Create Metadata Split

Builds `data/processed/metadata.json` with 70/15/15 split, balanced classes.

In [ ]:
import json, random
from pathlib import Path

random.seed(42)

raw_dir = Path('data/raw')
processed_dir = Path('data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

def collect_samples(folder: Path, label: int):
    return [
        {'path': str(Path(folder.name) / v.name), 'label': label}
        for v in sorted(folder.glob('*.mp4'))
    ]

real_samples = collect_samples(raw_dir / 'real', label=0)
fake_samples = collect_samples(raw_dir / 'fake', label=1)

# Balance classes by taking min count from each
min_count = min(len(real_samples), len(fake_samples))
print(f'Real: {len(real_samples)}, Fake: {len(fake_samples)} → balancing to {min_count} each')

random.shuffle(real_samples)
random.shuffle(fake_samples)
samples = real_samples[:min_count] + fake_samples[:min_count]
random.shuffle(samples)

total = len(samples)
train_end = int(total * 0.70)
val_end   = int(total * 0.85)

metadata = {
    'train': samples[:train_end],
    'val':   samples[train_end:val_end],
    'test':  samples[val_end:]
}

meta_path = processed_dir / 'metadata.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'[OK] Metadata saved to {meta_path}')
print(f'  Train: {len(metadata["train"])} | Val: {len(metadata["val"])} | Test: {len(metadata["test"])} samples')

## Cell 7 — Generate Kaggle Config (if not in repo)

In [ ]:
import yaml
from pathlib import Path

config_path = Path('configs/config_kaggle_full.yaml')
if config_path.exists():
    print(f'[OK] Using existing config: {config_path}')
else:
    print('[WARN] Config not found; writing inline default...')
    config_path.parent.mkdir(exist_ok=True)
    # Minimal fallback — ideally you should pull from the repo
    config = {
        'project_name': 'deepfake_detection',
        'experiment_name': 'kaggle_full_celebdf_v2',
        'seed': 42,
        'data': {'raw_dir': 'data/raw', 'processed_dir': 'data/processed',
                 'video_extensions': ['.mp4','.avi','.mov','.mkv'],
                 'audio_sample_rate': 16000, 'video_fps': 30,
                 'frame_size': 224, 'num_frames': 16, 'clip_duration': 2.0},
        'model': {
            'name': 'MultimodalAVDetector',
            'visual': {'backbone': 'dinov2_vits14', 'pretrained': True,
                       'freeze_backbone': False, 'embed_dim': 384},
            'temporal': {'type': 'video_swin', 'num_layers': 4,
                         'num_heads': 6, 'dropout': 0.1},
            'audio': {'backbone': 'wav2vec2', 'pretrained': True,
                      'freeze_backbone': True, 'embed_dim': 768},
            'frequency': {'wavelet_type': 'db8', 'levels': 3,
                          'bands': ['LL','LH','HL','HH']},
            'fusion': {'type': 'cross_attention', 'hidden_dim': 512,
                       'num_heads': 8, 'dropout': 0.15},
            'classifier': {'hidden_dims': [256, 128], 'dropout': 0.3, 'num_classes': 2}
        },
        'training': {
            'batch_size': 8, 'num_epochs': 30, 'learning_rate': 5e-5,
            'weight_decay': 1e-4, 'optimizer': 'adamw', 'scheduler': 'plateau',
            'warmup_epochs': 3,
            'loss': {'type': 'cross_entropy', 'label_smoothing': 0.1},
            'augmentation': {'horizontal_flip': True, 'color_jitter': True,
                             'gaussian_blur': True, 'compression_emulation': True,
                             'compression_quality': [70, 95]}
        },
        'validation': {'batch_size': 16, 'split_ratio': 0.15, 'eval_frequency': 1},
        'testing': {'batch_size': 16, 'tta': False, 'threshold': 0.5},
        'explainability': {'enabled': False, 'methods': ['gradcam'],
                           'save_visualizations': False,
                           'visualization_dir': 'logs/visualizations'},
        'logging': {'use_wandb': False, 'wandb_project': 'deepfake-detection',
                    'log_frequency': 20, 'checkpoint_dir': 'checkpoints/kaggle_full',
                    'save_frequency': 2, 'keep_last_n': 5},
        'hardware': {'device': 'cuda', 'num_workers': 4, 'pin_memory': True,
                     'mixed_precision': True, 'compile_model': False},
        'paths': {'pretrained_models': 'models/pretrained', 'outputs': 'outputs',
                  'logs': 'logs/kaggle_full'}
    }
    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    print(f'[OK] Config written to {config_path}')

# Print key settings
with open(config_path) as f:
    cfg = yaml.safe_load(f)
print(f'epochs={cfg["training"]["num_epochs"]}  '
      f'batch={cfg["training"]["batch_size"]}  '
      f'lr={cfg["training"]["learning_rate"]}  '
      f'frames={cfg["data"]["num_frames"]}  '
      f'frame_size={cfg["data"]["frame_size"]}')

## Cell 8 — Sanity Check: Load One Batch

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

import torch
from utils import DeepfakeDataset, get_transforms, load_config
from torch.utils.data import DataLoader

cfg = load_config('configs/config_kaggle_full.yaml')
val_transforms = get_transforms(cfg, mode='val')

train_ds = DeepfakeDataset(
    data_dir=cfg['data']['raw_dir'],
    metadata_file='data/processed/metadata.json',
    num_frames=cfg['data']['num_frames'],
    frame_size=cfg['data']['frame_size'],
    audio_sample_rate=cfg['data']['audio_sample_rate'],
    clip_duration=cfg['data']['clip_duration'],
    transform=val_transforms[0],
    audio_transform=val_transforms[1],
    mode='train'
)

loader = DataLoader(train_ds, batch_size=2, shuffle=False, num_workers=2)
batch = next(iter(loader))
print('[OK] Batch loaded successfully')
print(f'  video: {batch["video"].shape}  audio: {batch["audio"].shape}  label: {batch["label"]}')

## Cell 9 — Run Full Training

Change `RESUME_CHECKPOINT` to a real path to resume after a Kaggle session disconnect.

In [ ]:
import subprocess, sys

CONFIG       = 'configs/config_kaggle_full.yaml'
RESUME_CKPT  = None   # e.g. 'checkpoints/kaggle_full/checkpoint_epoch_10.pth'
NUM_EPOCHS   = None   # e.g. 40 to extend beyond config

cmd = [sys.executable, 'train.py', '--config', CONFIG]
if RESUME_CKPT:
    cmd += ['--resume', RESUME_CKPT]
if NUM_EPOCHS:
    cmd += ['--num-epochs', str(NUM_EPOCHS)]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

## Cell 10 — List Checkpoints (after training / for resume planning)

In [ ]:
import glob, os
ckpts = sorted(glob.glob('checkpoints/kaggle_full/*.pth'))
for c in ckpts:
    size_mb = os.path.getsize(c) / 1e6
    print(f'{c}  ({size_mb:.1f} MB)')

if not ckpts:
    print('No checkpoints found yet.')

## Cell 11 — Resume Example

Paste the latest checkpoint path above and re-run Cell 9, OR use this cell directly.

In [ ]:
# Example: resume from epoch 10 checkpoint and continue to epoch 30
RESUME_FROM = 'checkpoints/kaggle_full/checkpoint_epoch_10.pth'
EXTEND_TO   = 30

cmd = [
    sys.executable, 'train.py',
    '--config', 'configs/config_kaggle_full.yaml',
    '--resume', RESUME_FROM,
    '--num-epochs', str(EXTEND_TO)
]
print('Running:', ' '.join(cmd))
# Uncomment to actually run:
# subprocess.run(cmd, check=True)

## Cell 12 — Copy Best Model to /kaggle/working (for download)

In [ ]:
import shutil
best = Path('checkpoints/kaggle_full/best_model.pth')
if best.exists():
    dst = Path('/kaggle/working/best_model.pth')
    shutil.copy(best, dst)
    print(f'[OK] Best model copied to {dst}  ({dst.stat().st_size/1e6:.1f} MB)')
else:
    print('[WARN] best_model.pth not found. Run training first.')